# Wikipedia Popularity Table Generator

This notebook creates a popularity table by matching Wikipedia pageview data with article metadata.

## Configuration

In [22]:
import sys
import pathlib
sys.path.insert(0, str(pathlib.Path(".").resolve().parent.parent))

from config import DATA_DIR, CACHE_DIR

# ============== CONFIGURATION ==============
# Input: Pageviews data file (from Wikimedia dumps)
PAGEVIEWS_FILE = DATA_DIR / "raw_wikiPop_dumps" / "pageviews-202012-user"

# Input: Wikipedia dataset version (HuggingFace)
WIKIPEDIA_DATASET = "facebook/kilt_wikipedia"
WIKIPEDIA_VERSION = "2019-08-01"

# Output: Where to save the final popularity table
OUTPUT_FILE = DATA_DIR / "cleaned_wikiPop_dumps" / "popularity_table_202012.parquet"

# Filter: Wikipedia source to extract (e.g., "en.wikipedia")
WIKI_SOURCE = "en.wikipedia"
# ============================================

In [23]:
import pandas as pd
import os
from tqdm import tqdm

print(f"Processing {PAGEVIEWS_FILE}...")

# Optimized approach: Only read needed columns + filter in chunks to reduce memory
print(f"Loading and filtering data efficiently...")

# Get file size for progress bar
file_size = os.path.getsize(PAGEVIEWS_FILE)

chunks = []
chunksize = 1_000_000  # Process 1M rows at a time

# Wrap the file with tqdm for accurate progress tracking
with open(PAGEVIEWS_FILE, 'r') as f:
    with tqdm(total=file_size, unit='B', unit_scale=True, unit_divisor=1024, desc="Reading & Filtering") as pbar:
        # Track last position for progress updates
        last_pos = 0
        
        for chunk in pd.read_table(
            f,
            sep=" ",
            names=["source", "entity", "id", "device", "popularity", "features"],
            dtype={
                "source": str, 
                "entity": str,
                "id": str,
                "device": str, 
                "popularity": str,
                "features": str
            },
            usecols=["source", "id", "device", "popularity"],  # Keep all useful columns
            chunksize=chunksize,
            engine='c',  # Fastest parser
            low_memory=False,
            na_values=['', 'NA']
        ):
            # Filter immediately to reduce memory
            filtered = chunk[chunk["source"] == WIKI_SOURCE]
            if len(filtered) > 0:
                chunks.append(filtered)
            
            # Update progress bar based on actual file position
            current_pos = f.tell()
            pbar.update(current_pos - last_pos)
            last_pos = current_pos

# Concatenate all filtered chunks
print("\nConcatenating filtered chunks...")
wiki_data = pd.concat(chunks, ignore_index=True)

# Drop source column as it's no longer needed (all rows are same source)
wiki_data = wiki_data.drop(columns=["source"])

print(f"✓ Filtered to {len(wiki_data):,} rows for {WIKI_SOURCE}")
print(f"✓ DataFrame shape: {wiki_data.shape}")
print(f"✓ Memory usage: {wiki_data.memory_usage(deep=True).sum() / 1024**2:.1f} MB")

print("\nFirst 5 rows:")
wiki_data.head()

Processing /Users/cyro/Documents/VSC/PopularityBias/scripts/../data/raw_wikiPop_dumps/pageviews-202012-user...
Loading and filtering data efficiently...


Reading & Filtering: 100%|██████████| 11.6G/11.6G [01:09<00:00, 179MB/s]



Concatenating filtered chunks...
✓ Filtered to 32,878,120 rows for en.wikipedia
✓ DataFrame shape: (32878120, 3)
✓ Memory usage: 5697.0 MB

First 5 rows:


,id,device,popularity
0,5878274,desktop,25
1,5878274,mobile-web,3
2,7712754,desktop,687
3,7712754,mobile-web,371
4,3632887,desktop,265


## Step 2: Aggregate Pageviews by Page ID

Each page may have multiple rows (different devices, daily entries). We sum all pageviews per unique page ID.

**Why use ID instead of title?**
- IDs are stable (titles can change)
- No encoding issues (special characters, unicode)
- Unambiguous (no disambiguation needed)

In [25]:
# Aggregate pageviews by page ID
pageviews_df = wiki_data[["id", "popularity"]].groupby("id", as_index=False).agg({
    "popularity": "sum"
})

print(f"Unique pages: {len(pageviews_df):,}")
pageviews_df.head()

Unique pages: 10,730,932


,id,popularity
0,10,419
1,1000,1412151145225043793713525221089149684300142
2,10000,713122137634
3,100000,2
4,10000001,172


## Step 3: Load Wikipedia Article Metadata

We use the HuggingFace Wikipedia dataset to get article titles and IDs.
This allows us to match pageview IDs with human-readable titles.

In [26]:
from datasets import load_dataset 
import aiohttp

ds = load_dataset(
    path=WIKIPEDIA_DATASET,
    name=WIKIPEDIA_VERSION,
    split="full",
    cache_dir=CACHE_DIR,
    storage_options={'client_kwargs': {'timeout': aiohttp.ClientTimeout(total=3600)}}
)

print(f"Wikipedia articles: {len(ds):,}")
print(f"Columns: {ds.column_names}")

/Users/cyro/Documents/VSC/PopularityBias/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Loading dataset shards:   0%|          | 0/59 [00:00<?, ?it/s]

Wikipedia articles: 5,903,530
Columns: ['kilt_id', 'wikipedia_id', 'wikipedia_title', 'text', 'anchors', 'categories', 'wikidata_info', 'history']


## Step 5: Preview Results

Check the top and bottom articles by popularity.

In [27]:
# Convert to DataFrame for easier merging
sample_wiki_df = ds.take(5).to_pandas()
sample_wiki_df

,kilt_id,wikipedia_id,wikipedia_title,text,anchors,categories,wikidata_info,history
0,290,290,A,"{'paragraph': ['A ', 'A (named , plural ""As"", ...","{'paragraph_id': [1, 1, 1, 1, 1, 1, 1, 1, 2, 4...",ISO basic Latin letters,"{'description': '', 'enwikiquote_title': '', '...","{'pageid': 290, 'parentid': 906725792, 'revid'..."
1,39,39,Albedo,"{'paragraph': ['Albedo ', 'Albedo () (, meanin...","{'paragraph_id': [1, 1, 1, 1, 1, 1, 1, 1, 2, 2...","Electromagnetic radiation,Climate forcing,Radi...",{'description': 'ratio of reflected radiation ...,"{'pageid': 39, 'parentid': 906382973, 'revid':..."
2,316,316,Academy Award for Best Production Design,{'paragraph': ['Academy Award for Best Product...,"{'paragraph_id': [1, 1, 1, 1, 1, 1, 2, 4, 5], ...","Academy Awards,Best Art Direction Academy Awar...","{'description': 'award', 'enwikiquote_title': ...","{'pageid': 316, 'parentid': 904077177, 'revid'..."
3,330,330,Actrius,"{'paragraph': ['Actrius ', 'Actresses (Catalan...","{'paragraph_id': [1, 1, 1, 1, 3, 3, 3, 3, 5, 6...","1990s drama films,Catalan-language films,1997 ...","{'description': '1996 film by Ventura Pons', '...","{'pageid': 330, 'parentid': 883290952, 'revid'..."
4,332,332,Animalia (book),"{'paragraph': ['Animalia (book) ', 'Animalia i...","{'paragraph_id': [1, 1, 3, 3, 3, 3, 5, 5, 9, 9...","Alphabet books,Puzzle books,1986 children's bo...",{'description': 'illustrated children's book b...,"{'pageid': 332, 'parentid': 901621714, 'revid'..."


In [50]:
sample_wiki_df['text'].iloc[0]['paragraph']

array(['A\n',
       'A (named , plural "As", "A\'s", "a"s, "a\'s" or "aes") is the first letter and the first vowel of the modern English alphabet and the ISO basic Latin alphabet. It is similar to the Ancient Greek letter alpha, from which it derives. The uppercase version consists of the two slanting sides of a triangle, crossed in the middle by a horizontal bar. The lowercase version can be written in two forms: the double-storey a and single-storey ɑ. The latter is commonly used in handwriting and fonts based on it, especially fonts intended to be read by children, and is also found in italic type.\n',
       'In the English grammar, "a", and its variant "an", is an indefinite article.\n',
       'Section::::History.\n',
       'The earliest certain ancestor of "A" is aleph (also written \'aleph), the first letter of the Phoenician alphabet, which consisted entirely of consonants (for that reason, it is also called an abjad to distinguish it from a true alphabet). In turn, the anc

In [28]:
wiki_df = ds.select_columns(["wikipedia_id", "wikipedia_title"]).to_pandas()
wiki_df.head()

,wikipedia_id,wikipedia_title
0,290,A
1,39,Albedo
2,316,Academy Award for Best Production Design
3,330,Actrius
4,332,Animalia (book)


In [29]:
merged_df = wiki_df.merge(
    pageviews_df,
    left_on="wikipedia_id",
    right_on="id",
    how="left"
).drop(columns=["id"])

print(f"After merging {merged_df['popularity'].isna().sum()} rows have no popularity info. Eg., { (merged_df['popularity'].isna().sum() / len(merged_df)) * 100:.2f}%")
merged_df

After merging 133093 rows have no popularity info. Eg., 2.25%


,wikipedia_id,wikipedia_title,popularity
0,290,A,194434270125251211851486141110132352372
1,39,Albedo,7184271067238261766847961191
2,316,Academy Award for Best Production Design,1911212013232084283641
3,330,Actrius,14469
4,332,Animalia (book),4447725
...,...,...,...
5903525,5141786,Mongabay,4362512111
5903526,5141642,Normandie-class battleship,213693442132510623
5903527,5141759,New York State Route 129,13101
5903528,5141756,Novara,22623160414356831


In [30]:
merged_df['rank'] = merged_df['popularity'].rank(method='average', na_option='bottom', ascending=False)

merged_df.sort_values(by="rank")

#11144422

,wikipedia_id,wikipedia_title,popularity,rank
5737231,34541976,Christmas Eve with Johnny Mathis,999987,1.0
5528720,7320842,Fa jin,999961,2.0
3147168,38803140,JetSuite,999942,3.0
1795640,4768489,Houdini Live 2005: A Live History of Gluttony ...,99994015,4.0
4359679,4144636,Haplogroup B (mtDNA),999937,5.0
...,...,...,...,...
4423177,37959995,2013 Georgia State Panthers softball season,NaN,5836984.0
5395663,53574209,Kamal Public Sr. Secondary School,NaN,5836984.0
1425395,23159270,Adriana Tarasov,NaN,5836984.0
4423167,37959902,"Rasulabad, Iran",NaN,5836984.0


## Step 6: Save Output

Save the popularity table to a parquet file for efficient loading in other analyses.

In [31]:
merged_df.to_parquet(OUTPUT_FILE, index=False)
print(f"Saved to: {OUTPUT_FILE}")
print(f"File size: {os.path.getsize(OUTPUT_FILE) / 1e6:.1f} MB")

Saved to: /Users/cyro/Documents/VSC/PopularityBias/scripts/../data/cleaned_wikiPop_dumps/popularity_table_202012.parquet
File size: 197.6 MB
